# Test all-spine models for masked vertebral height

This notebook tests whether using every remaining vertebra improves prediction of one masked mean frontal height.

It compares nearest-neighbor interpolation, all-spine trend fitting, and a small masked Transformer. It predicts height only—no width, orientation, corners, or anatomical label.

**Research use only.** This reconstructs observed height from surrounding landmarks and is not clinically validated.

## Experiment flow

Ordered vertebral height sequence → replace one internal target with an interpolation token → retain every other height → fit simple trends and train a Transformer → compare all methods on the untouched test split.

Relative order is supplied only to distinguish measurements above and below the target. Anatomical levels are not required.

In [ ]:
# If packages are missing, uncomment and run once:
# %pip install -r ../../requirement.txt

## 1. Configuration

Use MAX_SAMPLES_PER_SPLIT for a quick trial. Leave it as None for the full experiment.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import math
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.interpolate import PchipInterpolator
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "dataset").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def weighted_mean(values, weights):
    return float(np.average(values, weights=weights))


def weighted_r2(actual, predicted, weights):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    actual_mean = weighted_mean(actual, weights)
    residual_sum = float(np.sum(weights * np.square(actual - predicted)))
    total_sum = float(np.sum(weights * np.square(actual - actual_mean)))
    if total_sum <= np.finfo(np.float64).eps:
        return 1.0 if residual_sum <= np.finfo(np.float64).eps else 0.0
    return 1.0 - residual_sum / total_sum


def weighted_within_percent(errors, threshold, weights):
    return 100.0 * weighted_mean(
        (np.asarray(errors) <= float(threshold)).astype(np.float64), weights
    )


def height_metrics(actual, baseline, predicted, weights):
    actual = np.asarray(actual, dtype=np.float64)
    baseline = np.asarray(baseline, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    baseline_error = np.abs(actual - baseline)
    model_error = np.abs(actual - predicted)
    baseline_mae = weighted_mean(baseline_error, weights)
    model_mae = weighted_mean(model_error, weights)
    return {
        "samples": len(actual),
        "baseline_mae": baseline_mae,
        "model_mae": model_mae,
        "baseline_rmse": float(np.sqrt(weighted_mean(np.square(baseline_error), weights))),
        "model_rmse": float(np.sqrt(weighted_mean(np.square(model_error), weights))),
        "baseline_r2": weighted_r2(actual, baseline, weights),
        "model_r2": weighted_r2(actual, predicted, weights),
        "model_median_absolute_error": float(np.median(model_error)),
        "model_p90_absolute_error": float(np.quantile(model_error, 0.90)),
        "model_p95_absolute_error": float(np.quantile(model_error, 0.95)),
        "baseline_within_5pct_reference_percent": weighted_within_percent(baseline_error, 0.05, weights),
        "model_within_5pct_reference_percent": weighted_within_percent(model_error, 0.05, weights),
        "baseline_within_10pct_reference_percent": weighted_within_percent(baseline_error, 0.10, weights),
        "model_within_10pct_reference_percent": weighted_within_percent(model_error, 0.10, weights),
        "relative_mae_improvement_percent": (
            100.0 * (baseline_mae - model_mae) / baseline_mae
            if baseline_mae > 0.0 else 0.0
        ),
    }

DATASET_ROOT = REPO_ROOT / "dataset/processed/masked_morphology_coco_nih_lumos"
OUTPUT_DIR = REPO_ROOT / "outputs/masked_morphology/all_spine_height_sequence_v1"
SEED = 20260814
MAX_SAMPLES_PER_SPLIT = None
MAX_SEQUENCE_LENGTH = 32
BATCH_SIZE = 512
EPOCHS = 60
PATIENCE = 12
LEARNING_RATE = 8e-4
SAVE_ARTIFACTS = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    torch.set_num_threads(4)

print("Device:", DEVICE)
print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

## 2. Construct variable-length masked sequences

Each token contains normalized height, relative rank, and an observed flag. The target token receives the interpolation height and observed=0. Padding tokens are hidden from attention.

The target output is the correction to interpolation. Normalization uses the median mean height of the same four immediate context vertebrae used by the earlier experiments.

In [ ]:
def build_sequence_split(split, limit=None):
    split_dir = DATASET_ROOT / split
    samples = pd.read_csv(split_dir / "masked_samples.csv", low_memory=False)
    if limit is not None:
        samples = samples.head(limit).copy()
    vertebrae = pd.read_csv(split_dir / "vertebrae.csv", low_memory=False)
    vertebrae = vertebrae.loc[vertebrae["geometry_valid"].astype(bool)]
    chains = {
        image_key: group.sort_values("chain_rank")
        for image_key, group in vertebrae.groupby("image_key", sort=False)
    }

    count = len(samples)
    tokens = np.zeros((count, MAX_SEQUENCE_LENGTH, 3), dtype=np.float32)
    padding_mask = np.ones((count, MAX_SEQUENCE_LENGTH), dtype=bool)
    target_indices = np.zeros(count, dtype=np.int64)
    actual = np.zeros(count, dtype=np.float32)
    baseline = np.zeros(count, dtype=np.float32)
    residual = np.zeros(count, dtype=np.float32)
    pchip_prediction = np.zeros(count, dtype=np.float32)
    quadratic_prediction = np.zeros(count, dtype=np.float32)

    for index, sample in enumerate(samples.itertuples(index=False)):
        chain = chains[sample.image_key]
        if len(chain) > MAX_SEQUENCE_LENGTH:
            raise ValueError(
                f"Chain length {len(chain)} exceeds MAX_SEQUENCE_LENGTH={MAX_SEQUENCE_LENGTH}."
            )
        annotation_ids = chain["coco_annotation_id"].to_numpy()
        matches = np.flatnonzero(annotation_ids == sample.target_annotation_id)
        if len(matches) != 1:
            raise ValueError(f"Target lookup failed for {sample.sample_id}")
        target_index = int(matches[0])
        ranks = chain["chain_rank"].to_numpy(dtype=np.float64)
        heights = (
            chain["mean_height"].to_numpy(dtype=np.float64)
            / float(sample.reference_height_px)
        )
        target_rank = ranks[target_index]
        target_actual = 0.5 * (
            float(sample.y_left_height_norm) + float(sample.y_right_height_norm)
        )
        target_baseline = 0.5 * (
            float(sample.baseline_left_height_norm)
            + float(sample.baseline_right_height_norm)
        )

        input_heights = heights.copy()
        input_heights[target_index] = target_baseline
        length = len(chain)
        tokens[index, :length, 0] = input_heights
        tokens[index, :length, 1] = (ranks - target_rank) / MAX_SEQUENCE_LENGTH
        tokens[index, :length, 2] = 1.0
        tokens[index, target_index, 2] = 0.0
        padding_mask[index, :length] = False
        target_indices[index] = target_index
        actual[index] = target_actual
        baseline[index] = target_baseline
        residual[index] = target_actual - target_baseline

        observed = np.arange(length) != target_index
        observed_ranks = ranks[observed]
        observed_heights = heights[observed]
        pchip_prediction[index] = float(
            PchipInterpolator(observed_ranks, observed_heights)(target_rank)
        )
        relative_ranks = observed_ranks - target_rank
        degree = min(2, len(relative_ranks) - 1)
        coefficients = np.polyfit(relative_ranks, observed_heights, degree)
        quadratic_prediction[index] = float(np.polyval(coefficients, 0.0))

    return {
        "samples": samples.reset_index(drop=True),
        "tokens": tokens,
        "padding_mask": padding_mask,
        "target_indices": target_indices,
        "actual": actual,
        "baseline": baseline,
        "residual": residual,
        "weights": samples["sample_weight"].to_numpy(dtype=np.float32),
        "pchip": pchip_prediction,
        "quadratic": quadratic_prediction,
    }

In [ ]:
sequence_data = {
    split: build_sequence_split(split, MAX_SAMPLES_PER_SPLIT)
    for split in ("train", "val", "test")
}
summary = pd.DataFrame({
    split: {
        "samples": len(data["samples"]),
        "groups": data["samples"]["group_id"].nunique(),
        "images": data["samples"]["image_key"].nunique(),
        "maximum_chain_length": int((~data["padding_mask"]).sum(axis=1).max()),
    }
    for split, data in sequence_data.items()
}).T
display(summary)

In [ ]:
for split, data in sequence_data.items():
    assert np.isfinite(data["tokens"]).all()
    assert np.isfinite(data["actual"]).all()
    assert np.isfinite(data["pchip"]).all()
    assert np.isfinite(data["quadratic"]).all()
    row_indices = np.arange(len(data["target_indices"]))
    assert np.all(data["tokens"][row_indices, data["target_indices"], 2] == 0.0)

group_sets = {
    split: set(data["samples"]["group_id"])
    for split, data in sequence_data.items()
}
assert group_sets["train"].isdisjoint(group_sets["val"])
assert group_sets["train"].isdisjoint(group_sets["test"])
assert group_sets["val"].isdisjoint(group_sets["test"])
print("Audit passed: masked target, finite sequences, and no group leakage.")

## 3. Inspect one masked sequence

In [ ]:
example_index = min(100, len(sequence_data["train"]["actual"]) - 1)
example = sequence_data["train"]
valid = ~example["padding_mask"][example_index]
target_index = example["target_indices"][example_index]
ranks = np.arange(valid.sum())
heights = example["tokens"][example_index, valid, 0]

fig, ax = plt.subplots(figsize=(9, 4))
observed = ranks != target_index
ax.plot(ranks[observed], heights[observed], "o-", label="Visible normalized heights")
ax.scatter([target_index], [example["actual"][example_index]], s=90, label="Hidden actual")
ax.scatter([target_index], [example["baseline"][example_index]], marker="x", s=90, label="Interpolation")
ax.scatter([target_index], [example["pchip"][example_index]], marker="s", s=60, label="All-spine PCHIP")
ax.scatter([target_index], [example["quadratic"][example_index]], marker="^", s=60, label="All-spine quadratic")
ax.set(xlabel="Chain rank", ylabel="Height / context median height")
ax.legend()
plt.show()

## 4. Evaluate non-learned baselines

Interpolation uses the nearest upper and lower vertebrae. PCHIP estimates local slopes from the remaining ordered sequence. Quadratic fitting uses the full visible height trend.

In [ ]:
def evaluate_method(split, method, predicted):
    data = sequence_data[split]
    return {
        "split": split,
        "method": method,
        **height_metrics(
            data["actual"], data["baseline"], predicted, data["weights"]
        ),
    }


baseline_rows = []
for split in ("val", "test"):
    data = sequence_data[split]
    baseline_rows.extend([
        evaluate_method(split, "nearest_interpolation", data["baseline"]),
        evaluate_method(split, "all_spine_pchip", data["pchip"]),
        evaluate_method(split, "all_spine_quadratic", data["quadratic"]),
    ])
baseline_metrics = pd.DataFrame(baseline_rows)
display(baseline_metrics[[
    "split", "method", "model_mae", "model_rmse",
    "model_within_5pct_reference_percent",
    "model_within_10pct_reference_percent", "model_r2",
]].round(4))

## 5. Define the masked Transformer

Attention can use all visible vertebrae regardless of chain length. The prediction head reads only the masked target token and outputs a correction to interpolation.

In [ ]:
class MaskedHeightTransformer(nn.Module):
    def __init__(self, d_model=32, heads=4, layers=2, dropout=0.10):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(3, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=2 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=False,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers,
            norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, tokens, padding_mask, target_indices):
        encoded = self.encoder(
            self.input_projection(tokens),
            src_key_padding_mask=padding_mask,
        )
        batch_indices = torch.arange(len(encoded), device=encoded.device)
        target_state = encoded[batch_indices, target_indices]
        return self.head(target_state).squeeze(-1)


model = MaskedHeightTransformer().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def make_loader(split, shuffle=False):
    data = sequence_data[split]
    dataset = TensorDataset(
        torch.from_numpy(data["tokens"]),
        torch.from_numpy(data["padding_mask"]),
        torch.from_numpy(data["target_indices"]),
        torch.from_numpy(data["residual"]),
        torch.from_numpy(data["weights"]),
    )
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )


loaders = {
    split: make_loader(split, shuffle=(split == "train"))
    for split in ("train", "val", "test")
}

## 6. Train using validation-only early stopping

The weighted Smooth L1 loss limits the influence of unusually large errors. The test split is not used for optimization or early stopping.

In [ ]:
def predict_residuals(model, loader):
    model.eval()
    outputs = []
    with torch.no_grad():
        for tokens, padding_mask, target_indices, _, _ in loader:
            prediction = model(
                tokens.to(DEVICE),
                padding_mask.to(DEVICE),
                target_indices.to(DEVICE),
            )
            outputs.append(prediction.cpu().numpy())
    return np.concatenate(outputs)


def weighted_validation_mae(model):
    predicted = predict_residuals(model, loaders["val"])
    data = sequence_data["val"]
    return float(np.average(np.abs(data["residual"] - predicted), weights=data["weights"]))

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=4, min_lr=1e-5
)
history = []
best_validation_mae = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    weighted_loss_sum = 0.0
    weight_sum = 0.0
    for tokens, padding_mask, target_indices, residual, weights in loaders["train"]:
        tokens = tokens.to(DEVICE)
        padding_mask = padding_mask.to(DEVICE)
        target_indices = target_indices.to(DEVICE)
        residual = residual.to(DEVICE)
        weights = weights.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        predicted = model(tokens, padding_mask, target_indices)
        per_sample_loss = nn.functional.smooth_l1_loss(
            predicted, residual, reduction="none", beta=0.05
        )
        loss = (per_sample_loss * weights).sum() / weights.sum()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        weighted_loss_sum += float((per_sample_loss * weights).sum().item())
        weight_sum += float(weights.sum().item())

    train_loss = weighted_loss_sum / weight_sum
    validation_mae = weighted_validation_mae(model)
    scheduler.step(validation_mae)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_mae": validation_mae,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })

    if validation_mae < best_validation_mae - 1e-5:
        best_validation_mae = validation_mae
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch={epoch:03d} train_loss={train_loss:.5f} "
            f"val_mae={validation_mae:.5f}"
        )
    if epochs_without_improvement >= PATIENCE:
        print("Early stopping at epoch", epoch)
        break

if best_state is None:
    raise RuntimeError("Training did not produce a checkpoint.")
model.load_state_dict(best_state)
print("Best validation MAE:", best_validation_mae)

In [ ]:
history_frame = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_frame["epoch"], history_frame["validation_mae"], label="Validation MAE")
ax.set(xlabel="Epoch", ylabel="Normalized residual MAE")
ax.legend()
plt.show()

## 7. Final held-out comparison

Within 5% means error no greater than 0.05 times the median height of the four immediate context vertebrae.

In [ ]:
transformer_residuals = {
    split: predict_residuals(model, loaders[split])
    for split in ("val", "test")
}
evaluation_rows = []
prediction_tables = []
for split in ("val", "test"):
    data = sequence_data[split]
    transformer_height = data["baseline"] + transformer_residuals[split]
    method_predictions = {
        "nearest_interpolation": data["baseline"],
        "all_spine_pchip": data["pchip"],
        "all_spine_quadratic": data["quadratic"],
        "masked_transformer": transformer_height,
    }
    for method, predicted in method_predictions.items():
        evaluation_rows.append(evaluate_method(split, method, predicted))

    table = data["samples"][[
        "sample_id", "group_id", "image_path", "source_dataset",
        "target_annotation_id", "target_chain_rank", "chain_count", "sample_weight",
    ]].copy()
    table.insert(1, "split", split)
    table["actual_mean_height_norm"] = data["actual"]
    for method, predicted in method_predictions.items():
        table[f"{method}_predicted"] = predicted
        table[f"{method}_absolute_error"] = np.abs(data["actual"] - predicted)
    prediction_tables.append(table)

metrics = pd.DataFrame(evaluation_rows)
predictions = pd.concat(prediction_tables, ignore_index=True)
display(metrics[[
    "split", "method", "model_mae", "model_rmse",
    "model_within_5pct_reference_percent",
    "model_within_10pct_reference_percent", "model_r2",
]].round(4))

In [ ]:
test_metrics = metrics.loc[metrics["split"].eq("test")].set_index("method")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
test_metrics["model_mae"].sort_values().plot.bar(ax=axes[0], color="#219ebc")
axes[0].set(title="Test normalized MAE", ylabel="Lower is better")
test_metrics[[
    "model_within_5pct_reference_percent",
    "model_within_10pct_reference_percent",
]].plot.bar(ax=axes[1], color=["#ffb703", "#219ebc"])
axes[1].set(title="Test tolerance accuracy", ylabel="Higher is better", ylim=(0, 100))
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 8. Save reproducible artifacts

In [ ]:
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_class": "MaskedHeightTransformer",
            "input_features": ["height_or_baseline", "relative_rank", "observed"],
            "max_sequence_length": MAX_SEQUENCE_LENGTH,
            "seed": SEED,
            "research_use_only": True,
        },
        OUTPUT_DIR / "masked_height_transformer.pt",
    )
    metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    history_frame.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
    manifest = {
        "task": "all-visible-context masked mean vertebral height prediction",
        "research_use_only": True,
        "dataset_root": str(DATASET_ROOT),
        "normalization": "median mean height of immediate offsets m2, m1, p1, p2",
        "target": "mean frontal height only",
        "device": str(DEVICE),
        "seed": SEED,
        "epochs_completed": len(history_frame),
        "best_validation_mae": best_validation_mae,
        "split_rows": {
            split: len(data["samples"]) for split, data in sequence_data.items()
        },
    }
    with (OUTPUT_DIR / "run_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
        file.write("\n")

    reloaded = MaskedHeightTransformer().to(DEVICE)
    checkpoint = torch.load(
        OUTPUT_DIR / "masked_height_transformer.pt",
        map_location=DEVICE,
        weights_only=True,
    )
    reloaded.load_state_dict(checkpoint["model_state_dict"])
    check = predict_residuals(reloaded, loaders["test"])
    assert np.isfinite(check).all()
    print("Saved and reloaded:", OUTPUT_DIR)

## Decision rule

Use the Transformer only if its held-out improvement is large enough to justify extra complexity and inference time. If it is similar to interpolation, prefer the simpler method.

A weak result also indicates an information limit: the remaining vertebral heights may not uniquely determine the hidden observed height.